# Laboratorio 7 — Spark MLlib
### CC3066 Data Science — Semestre II 2026
**Universidad del Valle de Guatemala — Facultad de Ingeniería**

**Integrantes del grupo**

| Integrante | Carné |
|---|---|
| Cindy Gualim | 21226 |
| Jose Donado |  |
| Daniela Ramírez |  |

---

Análisis de salarios de personas asalariadas con la base de **Personas** de la
Encuesta Nacional de Empleo e Ingresos Continua (ENEIC) del INE de Guatemala.
Se usan los cuatro trimestres de 2025 para desarrollo/entrenamiento y el
I trimestre de 2026 como prueba final.

## 1. Carga, armonización y calidad de datos

En esta sección:

1. Cargamos los cinco archivos de Excel identificando su procedencia.
2. Seleccionamos únicamente las columnas requeridas y homologamos sus tipos.
3. Unimos los cuatro archivos de 2025 con `unionByName`.
4. Documentamos faltantes, exclusiones por filtro y unicidad de la clave.
5. Guardamos los conjuntos preparados de 2025 y 2026 en Parquet.

> **Nota sobre el período:** la columna original `TRIMESTRE` **no** corresponde al
> trimestre calendario (I de 2025 trae el valor 2, y II de 2025 trae 3 y también 2
> en 175 registros). Por eso el período se deriva del **archivo de procedencia**,
> no de la columna, y la columna original se conserva solo para auditoría.

### 1.1 Configuración del entorno y sesión de Spark

In [ ]:
import os, glob

import pandas as pd
import numpy as np

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, IntegerType)

spark = (SparkSession.builder
         .appName("Lab7-ENEIC-SparkMLlib")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)

### 1.2 Rutas y mapeo de archivos → período

`periodo_archivo`, `anio_archivo` y `trimestre_calendario` se construyen a partir
del archivo de origen. `archivo_origen` se conserva para trazabilidad.

In [ ]:
DATA_DIR    = "data"      # carpeta con los .xlsx de la ENEIC (bases de Personas)
PARQUET_DIR = "parquet"   # salida intermedia
os.makedirs(PARQUET_DIR, exist_ok=True)

# periodo -> (año, trimestre calendario, uso previsto)
MAPEO_PERIODOS = {
    "2025T1": {"anio": 2025, "trimestre": 1, "uso": "entrenamiento"},
    "2025T2": {"anio": 2025, "trimestre": 2, "uso": "entrenamiento"},
    "2025T3": {"anio": 2025, "trimestre": 3, "uso": "entrenamiento"},
    "2025T4": {"anio": 2025, "trimestre": 4, "uso": "validacion"},
    "2026T1": {"anio": 2026, "trimestre": 1, "uso": "prueba final"},
}

print("Archivos disponibles en", DATA_DIR)
for f in sorted(glob.glob(os.path.join(DATA_DIR, "*.xls*"))):
    print("  -", os.path.basename(f))

El emparejamiento archivo → período se declara de forma **explícita** (los nombres de
archivo del INE varían), de modo que la asignación quede documentada y reproducible.

In [ ]:
# EDITAR con el nombre exacto de cada archivo descargado del INE
ARCHIVOS = {
    "2025T1": "ENEIC_I_2025_Personas.xlsx",
    "2025T2": "ENEIC_II_2025_Personas.xlsx",
    "2025T3": "ENEIC_III_2025_Personas.xlsx",
    "2025T4": "ENEIC_IV_2025_Personas.xlsx",
    "2026T1": "ENEIC_I_2026_Personas.xlsx",
}

for p, nombre in ARCHIVOS.items():
    ruta = os.path.join(DATA_DIR, nombre)
    print(f"{p}: {'OK   ' if os.path.exists(ruta) else 'FALTA'} {ruta}")

### 1.3 Columnas requeridas y tipos

| Original | Nombre analítico | Uso |
|---|---|---|
| P05D01 | salario_mensual | variable objetivo |
| P02A03 | edad | predictor numérico / clustering |
| P05C07A | antiguedad_anios | construcción de antigüedad |
| P05C07B | antiguedad_meses | construcción de antigüedad |
| P05H01A | horas_semanales | predictor / clustering |
| P03A03A | nivel_educativo | predictor categórico |
| P05C16 | categoria_ocupacional | filtro y predictor categórico |
| DOMINIO | dominio | predictor categórico |
| OCUPADOS | ocupado | filtro |
| NUM_HOGAR, NUM_PERSONA | (se conservan) | auditoría de registros |
| FACTOR | (se conserva) | diseño muestral |
| ANIO, TRIMESTRE | (se conservan) | auditoría de la fuente |

Un mismo código categórico puede llegar como número en un archivo y como texto en otro,
por lo que **se normaliza a texto** antes de unir.

In [ ]:
COLS_NUM = {                      # original -> analítico (numéricas)
    "P05D01":  "salario_mensual",
    "P02A03":  "edad",
    "P05C07A": "antiguedad_anios",
    "P05C07B": "antiguedad_meses",
    "P05H01A": "horas_semanales",
}
COLS_CAT = {                      # original -> analítico (categóricas, como texto)
    "P03A03A": "nivel_educativo",
    "P05C16":  "categoria_ocupacional",
    "DOMINIO": "dominio",
}
COLS_AUD = ["OCUPADOS", "NUM_HOGAR", "NUM_PERSONA", "FACTOR", "ANIO", "TRIMESTRE"]

COLS_ORIGINALES = list(COLS_NUM) + list(COLS_CAT) + COLS_AUD

ESQUEMA = StructType(
    [StructField(v, DoubleType(), True) for v in COLS_NUM.values()] +
    [StructField(v, StringType(), True) for v in COLS_CAT.values()] +
    [StructField("ocupado",     DoubleType(), True),
     StructField("NUM_HOGAR",   StringType(), True),
     StructField("NUM_PERSONA", StringType(), True),
     StructField("FACTOR",      DoubleType(), True),
     StructField("ANIO",        StringType(), True),
     StructField("TRIMESTRE",   StringType(), True),
     StructField("periodo_archivo",      StringType(),  True),
     StructField("anio_archivo",         IntegerType(), True),
     StructField("trimestre_calendario", IntegerType(), True),
     StructField("archivo_origen",       StringType(),  True)]
)

#### Función de carga

Cada archivo se lee **individualmente** con pandas/openpyxl (Spark no tiene lector nativo
de Excel), se convierte a DataFrame de Spark con tipos explícitos y se persiste en
Parquet. Procesarlos de uno en uno controla el consumo de memoria.

Criterios de tipificación:

- **Numéricas:** `pd.to_numeric(errors="coerce")`; los valores no evaluables quedan nulos
  y se **contabilizan** más adelante. No se imputa nada, y en particular no el salario.
- **Categóricas:** se normalizan a texto sin decimales sobrantes (`2.0` → `"2"`), se
  recorta el espacio en blanco y el vacío queda nulo para marcarse luego como
  `DESCONOCIDO`. El código educativo `0` ("ninguno") se preserva como categoría válida.

In [ ]:
def _norm_cat(serie: pd.Series) -> pd.Series:
    """Normaliza un código categórico a texto consistente: 2, 2.0 y ' 2 ' -> '2'."""
    def f(v):
        if pd.isna(v):
            return None
        if isinstance(v, (int, np.integer)):
            return str(int(v))
        if isinstance(v, (float, np.floating)) and float(v).is_integer():
            return str(int(v))
        s = str(v).strip()
        if s == "":
            return None
        try:                        # '2.0' escrito como texto
            fv = float(s)
            if fv.is_integer():
                return str(int(fv))
        except ValueError:
            pass
        return s
    return serie.map(f)


def cargar_periodo(periodo):
    """Lee un archivo, selecciona y tipifica columnas, y lo guarda en Parquet."""
    meta   = MAPEO_PERIODOS[periodo]
    nombre = ARCHIVOS[periodo]
    ruta   = os.path.join(DATA_DIR, nombre)

    pdf = pd.read_excel(ruta, engine="openpyxl")
    pdf.columns = [str(c).strip().upper() for c in pdf.columns]

    ausentes = [c for c in COLS_ORIGINALES if c not in pdf.columns]
    if ausentes:
        raise KeyError(f"{periodo}: faltan columnas {ausentes}")

    out = pd.DataFrame(index=pdf.index)
    for orig, nuevo in COLS_NUM.items():
        out[nuevo] = pd.to_numeric(pdf[orig], errors="coerce").astype("float64")
    for orig, nuevo in COLS_CAT.items():
        out[nuevo] = _norm_cat(pdf[orig])

    out["ocupado"]     = pd.to_numeric(pdf["OCUPADOS"], errors="coerce").astype("float64")
    out["NUM_HOGAR"]   = _norm_cat(pdf["NUM_HOGAR"])
    out["NUM_PERSONA"] = _norm_cat(pdf["NUM_PERSONA"])
    out["FACTOR"]      = pd.to_numeric(pdf["FACTOR"], errors="coerce").astype("float64")
    out["ANIO"]        = _norm_cat(pdf["ANIO"])
    out["TRIMESTRE"]   = _norm_cat(pdf["TRIMESTRE"])

    out["periodo_archivo"]      = periodo
    out["anio_archivo"]         = meta["anio"]
    out["trimestre_calendario"] = meta["trimestre"]
    out["archivo_origen"]       = nombre

    out = out[[f.name for f in ESQUEMA.fields]]
    out = out.astype(object).where(pd.notna(out), None)

    sdf = spark.createDataFrame(out, schema=ESQUEMA)
    destino = os.path.join(PARQUET_DIR, f"crudo_{periodo}")
    sdf.write.mode("overwrite").parquet(destino)
    return spark.read.parquet(destino)

In [ ]:
crudos = {}
conteos_originales = {}

for periodo in ARCHIVOS:
    crudos[periodo] = cargar_periodo(periodo)
    conteos_originales[periodo] = crudos[periodo].count()
    print(f"{periodo}: {conteos_originales[periodo]:,} registros cargados")

### 1.4 Unión de los cuatro archivos de 2025

Se usa `unionByName`, que empareja por **nombre** de columna y no por posición. Esto es
indispensable porque el archivo de IV de 2025 trae 302 columnas frente a 270 de los demás.
Como aquí ya seleccionamos el mismo subconjunto de columnas en todos los archivos los
esquemas coinciden; aun así se conserva `unionByName` por corrección y trazabilidad.

In [ ]:
periodos_2025 = ["2025T1", "2025T2", "2025T3", "2025T4"]

df_2025 = crudos[periodos_2025[0]]
for p in periodos_2025[1:]:
    df_2025 = df_2025.unionByName(crudos[p], allowMissingColumns=True)

df_2026 = crudos["2026T1"]

df_2025.cache()
df_2026.cache()
print(f"2025 unido  : {df_2025.count():,} registros")
print(f"2026T1 (test): {df_2026.count():,} registros")

#### Esquema y cinco registros de las columnas seleccionadas

In [ ]:
df_2025.printSchema()

In [ ]:
df_2025.select("periodo_archivo", "archivo_origen", "ANIO", "TRIMESTRE",
               "trimestre_calendario", "salario_mensual", "edad",
               "antiguedad_anios", "antiguedad_meses", "horas_semanales",
               "nivel_educativo", "categoria_ocupacional", "dominio",
               "ocupado", "FACTOR").show(5, truncate=False)

#### Verificación del período

Comprobamos que la columna original `TRIMESTRE` no coincide con el trimestre calendario,
lo que justifica derivar el período del archivo de procedencia. Debe verse el valor 2 en
2025T1, los valores 3 y 2 en 2025T2, y así sucesivamente.

In [ ]:
(df_2025.unionByName(df_2026)
        .groupBy("periodo_archivo", "anio_archivo", "trimestre_calendario", "TRIMESTRE")
        .count()
        .orderBy("periodo_archivo", "TRIMESTRE")
        .show(20, truncate=False))

### 1.5 Faltantes por variable, **antes** de aplicar los filtros

In [ ]:
def tabla_faltantes(df, etiqueta):
    total = df.count()
    cols = [c for c in df.columns
            if c not in ("periodo_archivo", "anio_archivo",
                         "trimestre_calendario", "archivo_origen")]
    conteos = df.select([
        F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in cols
    ]).collect()[0].asDict()

    out = (pd.DataFrame({"variable": list(conteos), "faltantes": list(conteos.values())})
             .assign(pct_faltantes=lambda d: (100 * d.faltantes / total).round(2))
             .sort_values("faltantes", ascending=False))
    out.insert(0, "conjunto", etiqueta)
    return out


faltantes = pd.concat([tabla_faltantes(df_2025, "2025 (train)"),
                       tabla_faltantes(df_2026, "2026T1 (test)")],
                      ignore_index=True)
faltantes

**Interpretación.** Los porcentajes altos en `salario_mensual`, `antiguedad_*` y
`horas_semanales` son esperables: esas preguntas solo aplican a personas ocupadas y
asalariadas, de modo que la mayor parte de esos "faltantes" en el archivo completo son en
realidad *no aplica* (la pregunta no correspondía a esa persona), no respuestas perdidas.
Por eso la tabla se calcula **antes** de filtrar: una vez restringidos a la población
analítica, el faltante residual sí representa no respuesta.

### 1.6 Construcción de la antigüedad

$$\text{antigüedad (años)} = \text{antiguedad\_anios} + \frac{\text{antiguedad\_meses}}{12}$$

In [ ]:
def derivar_antiguedad(df):
    return df.withColumn(
        "antiguedad",
        F.when(F.col("antiguedad_anios").isNotNull() & F.col("antiguedad_meses").isNotNull(),
               F.col("antiguedad_anios") + F.col("antiguedad_meses") / F.lit(12.0)))

df_2025 = derivar_antiguedad(df_2025)
df_2026 = derivar_antiguedad(df_2026)
df_2025.select("antiguedad_anios", "antiguedad_meses", "antiguedad").show(5)

### 1.7 Filtrado secuencial de la población analítica

Se aplica **siempre el mismo orden** y se contabiliza cuántos registros se excluyen en
cada paso. No se imputa el salario ni se eliminan valores extremos.

| # | Filtro |
|---|---|
| 1 | Edad finita y ≥ 15 |
| 2 | Ocupado (`OCUPADOS = 1`) |
| 3 | Asalariado (`P05C16` ∈ {1, 2, 3, 4}) |
| 4 | `salario_mensual` numérico, finito y estrictamente positivo |
| 5 | Antigüedad en años no negativa |
| 6 | Componente de meses entero entre 0 y 11 |
| 7 | Antigüedad calculada ≤ edad |
| 8 | Horas habituales > 0 y ≤ 168 |

In [ ]:
FILTROS = [
    ("1. edad finita y >= 15",
     lambda d: d.filter(F.col("edad").isNotNull() & ~F.isnan("edad")
                        & (F.col("edad") >= 15))),
    ("2. ocupado (OCUPADOS = 1)",
     lambda d: d.filter(F.col("ocupado") == 1)),
    ("3. asalariado (P05C16 in 1..4)",
     lambda d: d.filter(F.col("categoria_ocupacional").isin("1", "2", "3", "4"))),
    ("4. salario finito y > 0",
     lambda d: d.filter(F.col("salario_mensual").isNotNull()
                        & ~F.isnan("salario_mensual")
                        & (F.col("salario_mensual") > 0)
                        & (F.col("salario_mensual") < float("inf")))),
    ("5. antiguedad_anios >= 0",
     lambda d: d.filter(F.col("antiguedad_anios").isNotNull()
                        & (F.col("antiguedad_anios") >= 0))),
    ("6. antiguedad_meses entero 0..11",
     lambda d: d.filter(F.col("antiguedad_meses").isNotNull()
                        & (F.col("antiguedad_meses") >= 0)
                        & (F.col("antiguedad_meses") <= 11)
                        & (F.col("antiguedad_meses") == F.floor("antiguedad_meses")))),
    ("7. antiguedad <= edad",
     lambda d: d.filter(F.col("antiguedad") <= F.col("edad"))),
    ("8. horas_semanales en (0, 168]",
     lambda d: d.filter(F.col("horas_semanales").isNotNull()
                        & ~F.isnan("horas_semanales")
                        & (F.col("horas_semanales") > 0)
                        & (F.col("horas_semanales") <= 168))),
]


def aplicar_filtros(df, etiqueta):
    actual = df
    n_prev = actual.count()
    filas = [{"conjunto": etiqueta, "paso": "0. original",
              "registros": n_prev, "excluidos": 0}]
    for nombre, f in FILTROS:
        actual = f(actual)
        n = actual.count()
        filas.append({"conjunto": etiqueta, "paso": nombre,
                      "registros": n, "excluidos": n_prev - n})
        n_prev = n
    return actual, pd.DataFrame(filas)

In [ ]:
analitico_2025, traza_2025 = aplicar_filtros(df_2025, "2025 (train)")
analitico_2026, traza_2026 = aplicar_filtros(df_2026, "2026T1 (test)")

analitico_2025.cache()
analitico_2026.cache()

traza = pd.concat([traza_2025, traza_2026], ignore_index=True)
traza

#### Registros por archivo, antes y después de los filtros

In [ ]:
antes = (df_2025.unionByName(df_2026)
         .groupBy("periodo_archivo").count()
         .withColumnRenamed("count", "antes_filtros"))
despues = (analitico_2025.unionByName(analitico_2026)
           .groupBy("periodo_archivo").count()
           .withColumnRenamed("count", "despues_filtros"))

(antes.join(despues, "periodo_archivo", "left")
      .fillna(0, subset=["despues_filtros"])
      .withColumn("retencion_pct",
                  F.round(100 * F.col("despues_filtros") / F.col("antes_filtros"), 2))
      .orderBy("periodo_archivo")
      .show(truncate=False))

**Interpretación.** La caída más grande ocurre en los pasos 2 y 3: la base de Personas
incluye a toda la población del hogar, y solo una fracción es ocupada **y** asalariada.
Las exclusiones posteriores (pasos 4 a 8) son de calidad de dato y deben ser
comparativamente pequeñas; si alguna resultara grande, conviene revisarla antes de
modelar porque estaría recortando la población analítica de forma no aleatoria.

### 1.8 Normalización de categóricas: `DESCONOCIDO`

Los valores ausentes o no reconocidos frente al diccionario se representan como
`DESCONOCIDO`, **no** como cero. En particular, el código educativo `0` significa
"ninguno" y es una categoría válida que se conserva.

In [ ]:
# Códigos válidos según el diccionario de datos de la ENEIC.
# VERIFICAR estos rangos contra el diccionario del archivo antes de la entrega.
CODIGOS_VALIDOS = {
    "nivel_educativo":       [str(i) for i in range(0, 11)],   # 0 = ninguno
    "categoria_ocupacional": ["1", "2", "3", "4"],
    "dominio":               [str(i) for i in range(1, 9)],
}


def marcar_desconocido(df):
    for col, validos in CODIGOS_VALIDOS.items():
        df = df.withColumn(
            col,
            F.when(F.col(col).isNotNull() & F.col(col).isin(validos), F.col(col))
             .otherwise(F.lit("DESCONOCIDO")))
    return df


analitico_2025 = marcar_desconocido(analitico_2025)
analitico_2026 = marcar_desconocido(analitico_2026)

for col in CODIGOS_VALIDOS:
    print(f"--- {col} (2025) ---")
    analitico_2025.groupBy(col).count().orderBy(F.desc("count")).show(15, truncate=False)

> Si aparece un volumen notable de `DESCONOCIDO`, es señal de que el rango de códigos
> asumido arriba no coincide con el diccionario de datos y debe corregirse antes de
> modelar: no es un faltante real, sino un error de configuración.

### 1.9 Unicidad de la clave (`periodo_archivo`, `NUM_HOGAR`, `NUM_PERSONA`)

No se usa `dropDuplicates()`. Si aparecen claves repetidas se investiga si son
repeticiones **exactas** o registros en **conflicto**.

In [ ]:
CLAVE = ["periodo_archivo", "NUM_HOGAR", "NUM_PERSONA"]


def revisar_unicidad(df, etiqueta):
    dups = df.groupBy(*CLAVE).count().filter(F.col("count") > 1)
    n_claves = dups.count()
    n_filas = dups.agg(F.coalesce(F.sum("count"), F.lit(0))).collect()[0][0]
    print(f"[{etiqueta}] claves duplicadas: {n_claves:,} | filas involucradas: {n_filas:,}")
    if n_claves == 0:
        print("   la clave es única dentro de cada período")
        return None

    filas_dup = df.join(dups.select(*CLAVE), CLAVE, "inner")
    cols_cmp = [c for c in df.columns if c not in CLAVE]
    exactos = (filas_dup.groupBy(*CLAVE, *cols_cmp).count()
                        .filter(F.col("count") > 1)
                        .agg(F.coalesce(F.sum("count"), F.lit(0)))
                        .collect()[0][0])
    print(f"   filas que son repetición exacta : {exactos:,}")
    print(f"   filas en conflicto (difieren)   : {n_filas - exactos:,}")
    filas_dup.orderBy(*CLAVE).show(10, truncate=False)
    return filas_dup


dup_2025 = revisar_unicidad(analitico_2025, "2025 (train)")
dup_2026 = revisar_unicidad(analitico_2026, "2026T1 (test)")

Una repetición **exacta** es redundancia del archivo y puede documentarse y consolidarse
con criterio explícito. Un registro **en conflicto** (misma clave, valores distintos) es
un problema de identidad que no se resuelve borrando una fila al azar: o la clave no
identifica de forma única a la persona dentro del período, o hay un error de captura. En
ambos casos se documenta en lugar de ocultarse con `dropDuplicates()`.

In [ ]:
# La persona SÍ puede repetirse ENTRE períodos: la ENEIC es un panel con rotación.
(analitico_2025.groupBy("NUM_HOGAR", "NUM_PERSONA")
 .agg(F.countDistinct("periodo_archivo").alias("n_periodos"))
 .groupBy("n_periodos").count()
 .orderBy("n_periodos")
 .show())

El conteo anterior muestra en cuántos trimestres aparece cada combinación
hogar–persona. Que buena parte aparezca en más de un período es consistente con el
**diseño longitudinal con rotación** de la encuesta, y no constituye un error de datos.

### 1.10 Respuestas a las preguntas de la sección

**¿Por qué IV de 2025 no puede apilarse por posición de columnas con los otros archivos?**

Porque tiene **302 columnas** frente a las 270 de los demás. Un apilado por posición
(`union` de Spark, que empareja por índice) alinearía la columna *k* de un archivo con la
columna *k* del otro, y a partir de la primera columna adicional todas quedarían
desplazadas: los valores de una variable terminarían almacenados bajo el nombre de otra,
mezclando tipos y, en el peor caso, sin arrojar ningún error visible. `unionByName`
empareja por **nombre**, que es la única forma correcta cuando los esquemas difieren.
Adicionalmente, aquí preseleccionamos el mismo subconjunto de columnas en todos los
archivos, lo que hace la unión robusta a ese cambio de estructura.

**¿Qué diferencia existe entre un dato ausente porque la pregunta no corresponde y una
respuesta no registrada?**

El *no aplica* es estructural: la pregunta nunca se le hizo a esa persona porque un filtro
previo del cuestionario la excluía —a quien no es asalariado no se le pregunta el salario
de la ocupación principal—. Su ausencia es informativa y **no debe imputarse**, porque el
valor simplemente no existe. La *no respuesta* ocurre cuando la pregunta sí correspondía
pero la persona no contestó o el dato no se registró; ahí sí hay un valor real
desconocido, y su ausencia puede introducir sesgo si no es aleatoria. Operativamente los
distinguimos por el orden de filtrado: lo ausente **antes** de restringir a la población
asalariada es mayoritariamente *no aplica*; lo que sigue ausente **dentro** de esa
población es no respuesta.

**¿Por qué una persona observada en dos períodos no debe eliminarse como duplicado del
conjunto longitudinal?**

Porque la unidad de análisis es la **persona-período**, no la persona. La ENEIC es un
panel con rotación: por diseño una misma vivienda se entrevista en varios trimestres
consecutivos. Sus dos registros son observaciones distintas —distinto trimestre, y
posiblemente distinto salario, antigüedad y jornada— y borrar una destruiría justamente la
variación temporal que hace útil el panel, además de sesgar la composición de la muestra
hacia los hogares que rotan más rápido. El duplicado problemático es otro: la misma clave
**dentro del mismo período**, que es exactamente lo que verifica la sección 1.9.

**¿Por qué el número de registros de la base filtrada no representa a todos los
trabajadores del país?**

Por tres razones acumuladas. (i) Es una **muestra**, no un censo: cada registro representa
a muchas personas mediante `FACTOR`, y nuestros conteos son sin ponderar. (ii)
Restringimos deliberadamente la población a asalariados de 15 años o más con salario
positivo registrado, lo que excluye por construcción a trabajadores por cuenta propia,
patronos, familiares no remunerados y a quienes no reportaron salario. (iii) Los filtros
de calidad eliminan además registros con valores inválidos o no evaluables, que no son una
submuestra aleatoria. Por eso los resultados describen **los registros analizados** y no
deben presentarse como estimaciones oficiales de la población guatemalteca.

**¿Para qué se utilizaría `FACTOR`?**

`FACTOR` es el factor de expansión del diseño muestral: indica a cuántas personas de la
población representa cada registro. En un análisis **poblacional** se usaría para ponderar
totales, medias y medianas —por ejemplo, el salario mediano de los asalariados del país— y,
junto con los estratos y las unidades primarias de muestreo, para estimar errores estándar
e intervalos de confianza correctos. En este laboratorio lo **conservamos pero no lo
usamos**, para mantener un alcance uniforme y no ponderado; tampoco entra como predictor
en los modelos.

### 1.11 Guardado en Parquet

Se guardan por separado el conjunto preparado de 2025 y el de 2026. Estos son el único
insumo de las secciones siguientes, de modo que el notebook puede ejecutarse de principio
a fin sin depender de variables creadas manualmente.

In [ ]:
RUTA_2025 = os.path.join(PARQUET_DIR, "analitico_2025")
RUTA_2026 = os.path.join(PARQUET_DIR, "analitico_2026T1")

analitico_2025.write.mode("overwrite").parquet(RUTA_2025)
analitico_2026.write.mode("overwrite").parquet(RUTA_2026)

print(f"2025   -> {RUTA_2025}   ({spark.read.parquet(RUTA_2025).count():,} registros)")
print(f"2026T1 -> {RUTA_2026}   ({spark.read.parquet(RUTA_2026).count():,} registros)")

**Resumen de la sección 1.** Los cinco archivos se cargaron individualmente, se
tipificaron de forma explícita y se unieron por nombre de columna. El período se derivó
del archivo de procedencia y no de la columna `TRIMESTRE`, que se conserva únicamente para
auditoría. Se documentaron los faltantes previos al filtrado, el número de registros
excluidos en cada paso de un filtrado de orden fijo, y la unicidad de la clave
período–hogar–persona sin recurrir a `dropDuplicates()`. Los conjuntos preparados quedaron
guardados en Parquet y son el insumo de las secciones siguientes.